# Linear Trend Forecast — Course Opportunity Score

**Goal:** Replace the simple `trend_30d` (two-window difference) with a proper **OLS linear regression** on weekly posting counts.

### What this notebook does
For each qualifying job position (course topic), we:
1. Bucket `first_seen` dates into **weekly counts**
2. Fit an **OLS linear regression**: `weekly_count ~ week_number`
3. Extract **slope** (postings gained/lost per week) and **R²** (consistency of trend)
4. Project forward **N weeks** (configurable) to get a near-term forecast
5. Assign a human-readable **trend label**: `Growing` / `Declining` / `Stable` / `Insufficient data`

### How to integrate into your existing pipeline
Inside `score_topics()`, replace the old `trend_30d` block with:
```python
FORECAST_WEEKS = 8    # change this to 4, 8, or 12
pos = add_trend_forecast(pos, postings, forecast_weeks=FORECAST_WEEKS)
```
The output column is named dynamically, e.g. `forecast_4w`, `forecast_8w`, `forecast_12w`.

## 0. Imports

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

print('imports ready')

## 1. Configuration — adjust forecast window here

**This is the only cell you need to change** to control the forecast horizon.

| Value | Best for |
|-------|----------|
| `4`  | Real-time / live API data (RapidAPI daily feed) |
| `8`  | Semi-static snapshots (recommended default) |
| `12` | Long-range planning, static datasets like LinkedIn 1.3M |

In [ ]:
# ── Change this value to switch forecast horizon ───────────────────────────
FORECAST_WEEKS = 8       # options: 4, 8, 12 (or any positive integer)

# ── Thresholds (rarely need changing) ─────────────────────────────────────
R2_THRESHOLD    = 0.30   # min R² to confidently label as Growing/Declining
SLOPE_THRESHOLD = 1.0    # min |postings/week| to call it a trend

print(f'Forecast window : {FORECAST_WEEKS} weeks')
print(f'R2 threshold    : {R2_THRESHOLD}')
print(f'Slope threshold : +/-{SLOPE_THRESHOLD} postings/week')

## 2. Core function — linear trend for a single position

| Output | Meaning |
|--------|---------|
| `slope` | postings gained (or lost) per week |
| `r2` | trend consistency (0 = noisy, 1 = perfect line) |
| `forecast` | projected total postings over the next N weeks |
| `trend_label` | `Growing` / `Declining` / `Stable` / `Insufficient data` |

In [ ]:
def _linear_trend_for_position(
    dates: pd.Series,
    forecast_weeks: int = 8,
    min_weeks: int = 4,
    r2_threshold: float = R2_THRESHOLD,
    slope_threshold: float = SLOPE_THRESHOLD,
) -> dict:
    """
    Fit OLS linear regression on weekly posting counts for one position.

    Parameters
    ----------
    dates           : Series of datetime (first_seen) for this position
    forecast_weeks  : how many weeks ahead to project
    min_weeks       : minimum non-zero weeks required to fit the model
    r2_threshold    : minimum R2 to assign a Growing/Declining label
    slope_threshold : minimum |slope| to call it a trend
    """
    null_result = {
        'slope':       0.0,
        'r2':          0.0,
        'forecast':    np.nan,
        'trend_label': 'Insufficient data',
    }

    dates = dates.dropna()
    if len(dates) < min_weeks * 3:
        return null_result

    # Step 1: bucket into ISO weeks
    week_series = dates.dt.to_period('W').value_counts().sort_index()
    if len(week_series) < min_weeks:
        return null_result

    x = np.arange(len(week_series), dtype=float)   # week index: 0, 1, 2, ...
    y = week_series.values.astype(float)

    # Step 2: OLS closed-form  slope = sum((xi-xbar)(yi-ybar)) / sum((xi-xbar)^2)
    x_mean, y_mean = x.mean(), y.mean()
    ss_xx = ((x - x_mean) ** 2).sum()
    if ss_xx == 0:
        return null_result

    slope     = ((x - x_mean) * (y - y_mean)).sum() / ss_xx
    intercept = y_mean - slope * x_mean

    # Step 3: R2
    y_pred = slope * x + intercept
    ss_res = ((y - y_pred) ** 2).sum()
    ss_tot = ((y - y_mean) ** 2).sum()
    r2 = float(np.clip(1 - ss_res / ss_tot if ss_tot > 0 else 0.0, 0.0, 1.0))

    # Step 4: N-week forward forecast (floor at 0)
    last_x   = x[-1]
    forecast = sum(
        max(0.0, slope * (last_x + k) + intercept)
        for k in range(1, forecast_weeks + 1)
    )

    # Step 5: trend label
    if slope > slope_threshold and r2 >= r2_threshold:
        label = 'Growing'
    elif slope < -slope_threshold and r2 >= r2_threshold:
        label = 'Declining'
    else:
        label = 'Stable'

    return {
        'slope':       round(float(slope), 3),
        'r2':          round(r2, 3),
        'forecast':    round(forecast, 1),
        'trend_label': label,
    }

print('_linear_trend_for_position defined')

## 3. Batch function — all positions at once

The forecast column is **named dynamically** from `forecast_weeks`,
so switching 4 → 8 → 12 automatically renames it too (`forecast_4w` / `forecast_8w` / `forecast_12w`).

In [ ]:
def add_trend_forecast(
    pos: pd.DataFrame,
    postings: pd.DataFrame,
    forecast_weeks: int = FORECAST_WEEKS,
    r2_threshold: float = R2_THRESHOLD,
    slope_threshold: float = SLOPE_THRESHOLD,
) -> pd.DataFrame:
    """
    Compute linear trend for every qualifying position and merge into pos.

    Returns pos with four new columns:
        trend_slope      – OLS slope (postings/week)
        trend_r2         – R2 of the fit (0-1)
        forecast_{N}w    – projected postings over next N weeks
        trend_label      – Growing | Declining | Stable | Insufficient data
    """
    forecast_col = f'forecast_{forecast_weeks}w'

    # guard: no usable dates in dataset
    if not postings['first_seen'].notna().any():
        pos['trend_slope'] = 0.0
        pos['trend_r2']    = 0.0
        pos[forecast_col]  = np.nan
        pos['trend_label'] = 'Insufficient data'
        return pos

    qualifying = set(pos['search_position'].tolist())
    subset = postings[
        postings['search_position'].isin(qualifying)
        & postings['first_seen'].notna()
    ][['search_position', 'first_seen']]

    records = []
    for position, grp in subset.groupby('search_position'):
        result = _linear_trend_for_position(
            grp['first_seen'],
            forecast_weeks=forecast_weeks,
            r2_threshold=r2_threshold,
            slope_threshold=slope_threshold,
        )
        result['search_position'] = position
        records.append(result)

    if not records:
        pos['trend_slope'] = 0.0
        pos['trend_r2']    = 0.0
        pos[forecast_col]  = np.nan
        pos['trend_label'] = 'Insufficient data'
        return pos

    trend_df = (
        pd.DataFrame(records)
        .rename(columns={
            'slope':       'trend_slope',
            'r2':          'trend_r2',
            'forecast':    forecast_col,
            'trend_label': 'trend_label',
        })
    )

    pos = pos.merge(trend_df, on='search_position', how='left')
    pos['trend_slope'] = pos['trend_slope'].fillna(0.0)
    pos['trend_r2']    = pos['trend_r2'].fillna(0.0)
    pos['trend_label'] = pos['trend_label'].fillna('Insufficient data')

    print(f'Trend forecast added  ->  column: "{forecast_col}"')
    return pos

print('add_trend_forecast defined')

## 4. Smoke test — synthetic data

| Position | Shape | Expected label |
|----------|-------|----------------|
| Data Engineer | Steadily increasing | **Growing** |
| Project Manager | Flat with noise | **Stable** |
| Manual QA Tester | Steadily decreasing | **Declining** |

In [ ]:
rng = np.random.default_rng(42)

def _make_postings(position: str, trend: str) -> pd.DataFrame:
    base  = pd.Timestamp('2024-01-01')
    dates = []
    for week in range(20):
        if trend == 'growing':
            count = max(0, int(10 + week * 3 + rng.integers(-2, 3)))
        elif trend == 'declining':
            count = max(0, int(60 - week * 2 + rng.integers(-2, 3)))
        else:
            count = max(0, int(20 + rng.integers(-3, 4)))
        for _ in range(count):
            dates.append(base + pd.Timedelta(days=week * 7 + int(rng.integers(0, 7))))
    return pd.DataFrame({
        'search_position': position,
        'first_seen':      pd.to_datetime(dates),
        'job_link':        [f'{position}_{i}' for i in range(len(dates))],
        'job_type': 'On-site', 'job_level': 'Mid senior',
        'search_city': 'New York', 'search_country': 'US', 'company': 'TestCo',
    })

postings_test = pd.concat([
    _make_postings('Data Engineer',    'growing'),
    _make_postings('Project Manager',  'stable'),
    _make_postings('Manual QA Tester', 'declining'),
], ignore_index=True)

pos_test = postings_test.groupby('search_position').agg(
    volume=('job_link', 'count')
).reset_index()

result = add_trend_forecast(pos_test.copy(), postings_test)

forecast_col = f'forecast_{FORECAST_WEEKS}w'
result[['search_position', 'volume', 'trend_slope', 'trend_r2', forecast_col, 'trend_label']]

## 5. Compare all three forecast windows side-by-side

In [ ]:
comparison = pos_test[['search_position', 'volume']].copy()

for weeks in [4, 8, 12]:
    tmp = add_trend_forecast(pos_test.copy(), postings_test, forecast_weeks=weeks)
    comparison[f'forecast_{weeks}w'] = tmp[f'forecast_{weeks}w'].values

comparison['trend_slope'] = result['trend_slope'].values
comparison['trend_r2']    = result['trend_r2'].values
comparison['trend_label'] = result['trend_label'].values

comparison[['search_position', 'trend_slope', 'trend_r2', 'trend_label',
            'forecast_4w', 'forecast_8w', 'forecast_12w']]

## 6. Visualise — actual data + trend line + N-week forecast

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

PALETTE = {'Growing': '#2E7D32', 'Stable': '#E6A817', 'Declining': '#C62828'}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle(f'Linear Trend Fits  —  {FORECAST_WEEKS}-week forecast',
             fontweight='bold', fontsize=13)

for ax, (_, row) in zip(axes, result.iterrows()):
    pos_name = row['search_position']
    label    = row['trend_label']
    slope    = row['trend_slope']
    r2       = row['trend_r2']
    color    = PALETTE.get(label, '#888888')

    dates_pos   = postings_test[postings_test['search_position'] == pos_name]['first_seen']
    week_counts = dates_pos.dt.to_period('W').value_counts().sort_index()
    x = np.arange(len(week_counts))
    y = week_counts.values.astype(float)

    x_mean      = x.mean()
    slope_v     = ((x - x_mean) * (y - y.mean())).sum() / ((x - x_mean) ** 2).sum()
    intercept_v = y.mean() - slope_v * x_mean
    y_line      = slope_v * x + intercept_v

    x_fut = np.arange(len(x), len(x) + FORECAST_WEEKS)
    y_fut = np.array([max(0.0, slope_v * xi + intercept_v) for xi in x_fut])

    ax.bar(x, y, color=color, alpha=0.30, label='Actual (weekly)')
    ax.plot(x, y_line, color=color, lw=2, label=f'Trend ({slope:+.1f}/wk)')
    ax.plot(x_fut, y_fut, color=color, lw=2, ls='--',
            label=f'{FORECAST_WEEKS}-wk forecast')
    ax.axvline(len(x) - 0.5, color='grey', lw=1, ls=':', label='Today')
    ax.set_title(f'{pos_name}\n{label}  |  R2={r2:.2f}', fontsize=11, color=color)
    ax.set_xlabel('Week number')
    ax.set_ylabel('Postings')
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v)}'))

plt.tight_layout()
plt.show()

## 7. Integration guide

### 7a. Replace `trend_30d` in `score_topics()`

```python
# REMOVE this entire block
if postings["first_seen"].notna().any():
    max_dt = postings["first_seen"].max()
    w1 = postings[(postings["first_seen"] >= max_dt - pd.Timedelta(days=30))]
    w0 = postings[
        (postings["first_seen"] < max_dt - pd.Timedelta(days=30))
        & (postings["first_seen"] >= max_dt - pd.Timedelta(days=60))
    ]
    c1 = w1.groupby("search_position")["job_link"].count()
    c0 = w0.groupby("search_position")["job_link"].count()
    pos["trend_30d"] = pos["search_position"].map((c1 - c0).fillna(0)).fillna(0).astype(float)
else:
    pos["trend_30d"] = 0.0

# ADD these two lines instead
FORECAST_WEEKS = 8    # change to 4, 8, or 12
pos = add_trend_forecast(pos, postings, forecast_weeks=FORECAST_WEEKS)
```

### 7b. Update display columns

```python
forecast_col = f'forecast_{FORECAST_WEEKS}w'   # e.g. 'forecast_8w'

["rank", "course_topic", "course_opportunity_score", "opportunity_label",
 "volume", "salary_proxy", "breadth_score",
 "trend_slope", "trend_r2", forecast_col, "trend_label"]
```

### 7c. Streamlit sidebar slider (optional)

```python
FORECAST_WEEKS = st.sidebar.select_slider(
    'Forecast window (weeks)',
    options=[4, 8, 12],
    value=8,
)
ranked = add_trend_forecast(pos, postings, forecast_weeks=FORECAST_WEEKS)
```

### 7d. Column reference

| Column | Type | Meaning |
|--------|------|---------|
| `trend_slope` | float | Postings gained or lost per week (`+3.0` = 3 more/week) |
| `trend_r2` | float 0-1 | Trend consistency (0 = noisy, 1 = perfect line) |
| `forecast_{N}w` | float | Projected total postings over next N weeks |
| `trend_label` | string | `Growing` / `Declining` / `Stable` / `Insufficient data` |